In [0]:
# Import PySpark functions and types for column operations and transformations
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import trim, col

### Read from Bronze Table

In [0]:
# Read orders data from the bronze Delta table into a Spark DataFrame
df_orders = spark.table("retail_catalog.bronze.orders")

In [0]:
# Count null values in each column of the orders DataFrame for initial data quality assessment
from pyspark.sql.functions import when, count, col

nulls = df_orders.select([count(when(col(c).isNull(), c)).alias(c) for c in df_orders.columns])
display(nulls)

order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,0,0,0,160,1783,2965,0


In [0]:
# Print schema to review column types and inspect a preview of the raw data
df_orders.printSchema()
display(df_orders.limit(5))

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)



order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02T10:56:33.000Z,2017-10-02T11:07:15.000Z,2017-10-04T19:55:00.000Z,2017-10-10T21:25:13.000Z,2017-10-18T00:00:00.000Z
53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24T20:41:37.000Z,2018-07-26T03:24:27.000Z,2018-07-26T14:31:00.000Z,2018-08-07T15:27:45.000Z,2018-08-13T00:00:00.000Z
47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08T08:38:49.000Z,2018-08-08T08:55:23.000Z,2018-08-08T13:50:00.000Z,2018-08-17T18:06:29.000Z,2018-09-04T00:00:00.000Z
949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18T19:28:06.000Z,2017-11-18T19:45:59.000Z,2017-11-22T13:39:59.000Z,2017-12-02T00:28:42.000Z,2017-12-15T00:00:00.000Z
ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13T21:18:39.000Z,2018-02-13T22:20:29.000Z,2018-02-14T19:46:34.000Z,2018-02-16T18:17:02.000Z,2018-02-26T00:00:00.000Z


### Transformations

Trimming strings

In [0]:
# Trim whitespace from all string columns for standardized, clean values
for field in df_orders.schema.fields:
    if isinstance(field.dataType, StringType):
        df_orders= df_orders.withColumn(field.name, trim(col(field.name)))

In [0]:
# Apply typecasting and value normalization to key columns:
# - Cast IDs to string, status to lowercase string
# - Convert timestamps to standard datetime format
from pyspark.sql.functions import to_timestamp, col
from pyspark.sql import functions as F

df_orders_clean = (
    df_orders
    # Primary key
    .withColumn("order_id", F.col("order_id").cast("string"))

    # Customer ID: force string
    .withColumn(
        "customer_id",
        F.col("customer_id").cast("string")
    )

    # Order Status: force string and lowercase
    .withColumn(
        "order_status",
        F.lower(F.col("order_status").cast("string"))
    )

    # Timestamps: parse consistently with expected formats
    .withColumn(
        "order_purchase_timestamp",
        F.to_timestamp(
            F.expr("try_to_timestamp(order_purchase_timestamp, 'dd-MM-yyyy HH:mm')")
        )
    )
    .withColumn(
        "order_approved_at",
        F.to_timestamp(
            F.expr("try_to_timestamp(order_approved_at, 'dd-MM-yyyy HH:mm')")
        )
    )
    .withColumn(
        "order_delivered_carrier_date",
        F.to_timestamp(
            F.expr("try_to_timestamp(order_delivered_carrier_date, 'dd-MM-yyyy HH:mm')")
        )
    )
    .withColumn(
        "order_delivered_customer_date",
        F.to_timestamp(
            F.expr("try_to_timestamp(order_delivered_customer_date, 'dd-MM-yyyy HH:mm')")
        )
    )    
    .withColumn(
        "order_estimated_delivery_date",
        F.to_timestamp(
            F.expr("try_to_timestamp(order_estimated_delivery_date, 'dd-MM-yyyy')")
        )
    )
)

df_orders_clean.show(5)

+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|            order_id|         customer_id|order_status|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|e481f51cbdc54678b...|9ef432eb625129730...|   delivered|     2017-10-02 10:56:33|2017-10-02 11:07:15|         2017-10-04 19:55:00|          2017-10-10 21:25:13|          2017-10-18 00:00:00|
|53cdb2fc8bc7dce0b...|b0830fb4747a6c6d2...|   delivered|     2018-07-24 20:41:37|2018-07-26 03:24:27|         2018-07-26 14:31:00|          2018-08-07 15:27:45|          2018-08-13 00:00:00|
|47770eb9100c2d0c4...|41ce2a54c0b03bf34...|  

In [0]:
# Print schema after transformation to ensure all columns are cast and parsed correctly
df_orders_clean.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)



In [0]:
# Filter out rows missing key fields for data integrity
# Only retain rows with order_id, customer_id, and purchase timestamp
from pyspark.sql.functions import col
df_orders_clean = df_orders_clean.filter(
    col("order_id").isNotNull() &
    col("customer_id").isNotNull() &
    col("order_purchase_timestamp").isNotNull()
)

In [0]:
# Write the cleaned orders DataFrame to the silver Delta layer (overwrite any previous data)
df_orders_clean.write.mode("overwrite").format("delta").saveAsTable("retail_catalog.silver.orders")